# Quant-Arena 方法迁移：现役 Stage1 信号到原生账户

**假设**：论文的“完整研究轨迹与最终账户”评测方法可以只读接入现役 f696 分配器 + b472 动作链的冻结研究账户，揭示目标到执行的传递缺口，但 A 股 Level-2 因子与择时阈值不迁移。

数据来自 `plan.json` 固定的三个 Qlib Recorder；0bp 指额外滑点为零，仍扣冻结 IBKR Tiered 费用。2023–2026 历史均已复用，任何结果只是诊断，不能构成新的独立验证。

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.analysis.decision_bridge import build_decision_bridge
from src.data.nas_market_data import archive_nas_recorder

root = Path.cwd().resolve()
card = root / "research/quant_arena_stage1_transfer_20261022"
plan_path = card / "plan.json"
plan = json.loads(plan_path.read_text(encoding="utf-8"))
plan_sha256 = hashlib.sha256(plan_path.read_bytes()).hexdigest()
assert hashlib.sha256((root / plan["release_path"]).read_bytes()).hexdigest() == plan["release_sha256"]
qlib.init(provider_uri=str(root / plan["provider_uri"]), region="us", kernels=1)
objects, source_audits = {}, {}
for role in ("parent", "action", "raw"):
    objects[role], source_audits[role] = _source(root, plan["sources"][role], plan["artifacts"][role])
parent = objects["parent"]["native/A_active_research_replay_0"]
benchmark = objects["raw"]["native/raw_SOXX_0"]
path = objects["parent"]["paths/A_active_research_replay_0"]
frame = objects["raw"]["inputs/frame"].reset_index(drop=True)
market = objects["action"]["inputs/market"].reset_index(drop=True)
source_target = objects["action"]["inputs/source_targets"]["frozen_current"]
decision_target = path["targets"].reset_index(drop=True)
authority = parent["authority"].reset_index(drop=True)
soxx = benchmark["authority"].reset_index(drop=True)
assert len(authority) == plan["sessions"]
assert parent["audit"]["status"] == benchmark["audit"]["status"] == "passed"
for other in (frame, market, decision_target, soxx):
    pd.testing.assert_frame_equal(authority[["signal_date", "trade_date", "exit_date"]], other[["signal_date", "trade_date", "exit_date"]]) if "exit_date" in other else pd.testing.assert_frame_equal(authority[["signal_date", "trade_date"]], other[["signal_date", "trade_date"]])
pd.testing.assert_index_equal(source_target.index, pd.DatetimeIndex(frame.signal_date))
assert pd.to_datetime(frame.feature_available_at).le(pd.to_datetime(frame.decision_at)).all()
assert pd.to_datetime(market.return_available_at).gt(pd.to_datetime(market.decision_at)).all()
calendar = pd.DatetimeIndex(D.calendar(start_time=frame.trade_date.min(), end_time=frame.exit_date.max(), freq="day"))
assert calendar.equals(pd.DatetimeIndex([*pd.to_datetime(frame.trade_date), pd.Timestamp(frame.exit_date.iloc[-1])]))
print({"source_status": "verified", "sessions": len(authority), "calendar_rows": len(calendar), "plan_sha256": plan_sha256})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[85051:MainThread](2026-10-22 20:13:17,872) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[85051:MainThread](2026-10-22 20:13:17,874) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[85051:MainThread](2026-10-22 20:13:17,874) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


{'source_status': 'verified', 'sessions': 944, 'calendar_rows': 945, 'plan_sha256': '813ff00e929e54b91ef052aae0471e8591cb948d363c7f173c84fdbab2e8c8e6'}


## 冻结模型的信号—动作—账户桥

`source_target` 为模型源目标；`projected_target` 在此绑定为动作链的决策目标，差值含风险和 HOLD 处理，不能单独归因。退出时持仓经过价格漂移，因此不伪称为成交后即时权重；这一阶段由桥报告为 unknown。按完整原生日历保留所有日期。

In [2]:
trace = pd.DataFrame({"trade_date": pd.to_datetime(authority.trade_date)})
for prefix, values in (("source", source_target), ("decision", decision_target)):
    for asset in ("soxl", "soxx", "cash"):
        trace[f"{prefix}_{asset}"] = values[f"{asset}_weight"].to_numpy(float)
order_dates = pd.to_datetime(parent["orders"].trade_date)
trace["order_count"] = order_dates.value_counts().reindex(trace.trade_date, fill_value=0).to_numpy(int)
trace["fees_usd"] = (authority.broker_and_external_fees_usd + authority.slippage_reserve_usd).to_numpy(float)
bridge = build_decision_bridge(
    trace, authority, native_audit=parent["audit"],
    source_identity={role: plan["sources"][role]["recorder_id"] for role in plan["sources"]},
    fields={"source_target": [f"source_{asset}" for asset in ("soxl", "soxx", "cash")],
            "projected_target": [f"decision_{asset}" for asset in ("soxl", "soxx", "cash")],
            "order_count": "order_count", "fees_usd": "fees_usd"},
    native_calendar=calendar,
)
relative_log = np.log1p(authority.strategy_return.to_numpy(float)) - np.log1p(soxx.strategy_return.to_numpy(float))
relative_wealth = float(np.exp(relative_log.sum()))
assert np.isclose(relative_wealth, float(authority.equity.iloc[-1] / soxx.equity.iloc[-1]), rtol=1e-9)
rolling = pd.DataFrame({"first_trade": trace.trade_date.iloc[:len(trace)-125].to_numpy(),
                        "last_trade": trace.trade_date.iloc[125:].to_numpy(),
                        "relative_wealth": np.exp(pd.Series(relative_log).rolling(126).sum().iloc[125:].to_numpy())})
annual = objects["parent"]["analysis/annual_metrics"].query("family == 'A_active_research_replay' and slippage_bps == 0").copy()
quarterly = objects["parent"]["analysis/quarterly_metrics"].query("family == 'A_active_research_replay' and slippage_bps == 0").copy()
source_rolling = objects["parent"]["analysis/rolling_summary"].query("family == 'A_active_research_replay' and slippage_bps == 0").iloc[0]
assert len(rolling) == int(source_rolling.windows) == 819
assert np.isclose(float((rolling.relative_wealth > 1).mean()), float(source_rolling.positive_fraction), atol=1e-10)
assert int(trace.order_count.sum()) == len(parent["orders"])
summary_metrics = {
    "stage1_0bp_terminal_wealth": float(authority.equity.iloc[-1]),
    "soxx_0bp_terminal_wealth": float(soxx.equity.iloc[-1]),
    "relative_wealth_ratio": relative_wealth,
    "source_risky_weight_mean_pct": float(100 * trace[["source_soxl", "source_soxx"]].sum(axis=1).mean()),
    "decision_risky_weight_mean_pct": float(100 * trace[["decision_soxl", "decision_soxx"]].sum(axis=1).mean()),
    "exit_cash_weight_mean_pct": float(100 * authority.ending_cash_weight.mean()),
    "source_to_decision_l1_mean": float(bridge["summary"]["source_target_to_projected_target_l1_mean"]),
    "open_execute_sessions": int(authority.execute_open_target.sum()),
    "confirmed_research_orders": int(trace.order_count.sum()),
    "order_charges_usd": float(trace.fees_usd.sum()),
    "complete_years": int(annual.complete_period.sum()),
    "positive_complete_years": int(annual.loc[annual.complete_period, "relative_wealth"].gt(1).sum()),
    "positive_rolling126_fraction": float((rolling.relative_wealth > 1).mean()),
    "rolling126_count": int(len(rolling)),
    "unknown_stages": bridge["summary"]["unknown_stages"],
    "economic_attribution_components": "needs_data: this parent book has no independent_replay per-session artifact; do not synthesize it",
}
print(summary_metrics)

{'stage1_0bp_terminal_wealth': 3.2968633192729415, 'soxx_0bp_terminal_wealth': 4.805878111063438, 'relative_wealth_ratio': 0.6860064369263443, 'source_risky_weight_mean_pct': 59.24951319963419, 'decision_risky_weight_mean_pct': 51.765788387831954, 'exit_cash_weight_mean_pct': 48.193693962829514, 'source_to_decision_l1_mean': 0.18551303070122932, 'open_execute_sessions': 797, 'confirmed_research_orders': 1275, 'order_charges_usd': 1558.5566298956603, 'complete_years': 2, 'positive_complete_years': 1, 'positive_rolling126_fraction': 0.31135531135531136, 'rolling126_count': 819, 'unknown_stages': ['postfill_weights'], 'economic_attribution_components': 'needs_data: this parent book has no independent_replay per-session artifact; do not synthesize it'}


## 原生 Recorder、复核与结论

本卡不拟合、不重放账户、不改变现役 SIM release。年度和季度使用来源卡已验证的完整期标签；逐日桥和连续 126 日相对财富写入新诊断 Recorder。论文的 A 股 Top-75、Level-2 和 5/20 日现金开关不构成本模型的新特征。

In [3]:
import yaml
tracking = card / "qlib_recorder"
R.set_uri(str(tracking))
with R.start(experiment_name=plan["experiment"], recorder_name="frozen_stage1_account_bridge", uri=str(tracking)):
    recorder = R.get_recorder()
    recorder.log_params(plan_sha256=plan_sha256, new_model_fits=0, new_account_replays=0,
                        parent_recorder_id=plan["sources"]["parent"]["recorder_id"])
    recorder.save_objects(artifact_path="inputs", plan=plan, source_audits=source_audits)
    recorder.save_objects(artifact_path="analysis", daily_bridge=bridge["daily"],
                          bridge_periods=bridge["periods"], relative_rolling126=rolling,
                          native_annual=annual, native_quarterly=quarterly,
                          summary_metrics=summary_metrics)
    recorder.log_metrics(relative_wealth_ratio=relative_wealth,
                         positive_rolling126_fraction=summary_metrics["positive_rolling126_fraction"],
                         source_to_decision_l1_mean=summary_metrics["source_to_decision_l1_mean"])
recorder_id = recorder.id
experiment_id = str(recorder.experiment_id)
recorder_dir = tracking / experiment_id / recorder_id
artifacts = recorder_dir / "artifacts"
manifest = {"recorder_id": recorder_id, "entries": [
    {"path": str(p.relative_to(card)), "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "bytes": p.stat().st_size}
    for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card / "verification_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
nas_evidence = archive_nas_recorder(card / "nas_binding.json", workspace_root=root, card=card,
                                    recorder_id=recorder_id, experiment_id=experiment_id)
summary = {
    "experiment": plan["experiment"], "plan_sha256": plan_sha256,
    "decision": plan["output_decision"], "objective_achieved": False,
    "new_model_fits": 0, "new_account_replays": 0,
    "parent_release": plan["parent"], "source_identities": plan["sources"],
    "provider_uri": str(root / plan["provider_uri"]), "region": "us",
    "recorder_id": recorder_id, "experiment_id": experiment_id,
    "recorder_uri": str(tracking), "recorder_local_dir": str(recorder_dir),
    "artifact_policy": {"policy": "slim_card_recorder_first", "workflow_config": "workflow.yaml",
                        "recorder_only_artifacts": ["inputs/*", "analysis/daily_bridge", "analysis/bridge_periods",
                                                    "analysis/relative_rolling126", "analysis/native_annual", "analysis/native_quarterly"]},
    "metrics": summary_metrics, "nas_evidence": nas_evidence,
    "limitations": ["All 944 dates are reused research history, not independent validation.",
                    "A is the frozen Stage1 model chain under the research 09:47 execution clock, not live 09:31 SIM fills.",
                    "End-of-interval weights are not postfill weights; the bridge leaves that stage unknown.",
                    "Relative wealth and risk gates remain the contract decision; signal scores are diagnostic."]}
(card / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(card / "workflow.yaml").write_text(yaml.safe_dump({"experiment": plan["experiment"],
    "provider_uri": summary["provider_uri"], "region": "us", "recorder_uri": summary["recorder_uri"],
    "plan": "plan.json", "artifact_policy": summary["artifact_policy"]}, sort_keys=False, allow_unicode=True), encoding="utf-8")
annual_lines = [f"{r.period}: relative wealth {r.relative_wealth:.4f}, complete={bool(r.complete_period)}"
                for r in annual.itertuples()]
conclusion = "# Quant-Arena 方法迁移：Stage1 冻结模型诊断结论\n\n" +     f"- 执行：核对三个固定 Qlib Recorder、944 个原生日历会话；新拟合 0、新账户 0；诊断 Recorder `{recorder_id}`。\n" +     f"- 0bp（仍扣 IBKR Tiered 费用）账户：Stage1 财富 {summary_metrics['stage1_0bp_terminal_wealth']:.4f}，SOXX {summary_metrics['soxx_0bp_terminal_wealth']:.4f}，相对财富 {relative_wealth:.4f}。\n" +     f"- 年度：{'；'.join(annual_lines)}。126日窗口 {len(rolling)} 个，正超额比例 {summary_metrics['positive_rolling126_fraction']:.1%}。\n" +     f"- 信号到决策目标平均 L1 距离 {summary_metrics['source_to_decision_l1_mean']:.4f}；源风险权重均值 {summary_metrics['source_risky_weight_mean_pct']:.1f}%，决策目标 {summary_metrics['decision_risky_weight_mean_pct']:.1f}%，退出现金 {summary_metrics['exit_cash_weight_mean_pct']:.1f}%。这些不能解读为可消除损失。\n" +     "- 缺口：无独立逐日 replay 对象，不能在本卡重建七项精确归因；成交后即时权重未知，退出持仓不可替代。\n" +     "- 决策：`diagnostic_only_no_model_change_no_promotion`。论文未提供与当前三资产同任务的新信息；现役 SIM 不变。风险：复用历史、来源 09:47 与 SIM 09:31 时钟不同、未来数据首次可用性仍需另证。下一步是只对新增且 PIT 合格的特征冻结信息门，再做完整 0bp 账户；本卡不是新候选或晋升证据。\n"
(card / "conclusion.md").write_text(conclusion, encoding="utf-8")
print({"recorder_id": recorder_id, "decision": summary["decision"], "nas_sha256": nas_evidence["sha256"]})

[85051:MainThread](2026-10-22 20:13:18,766) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name quant_arena_stage1_transfer_20261022.


[85051:MainThread](2026-10-22 20:13:18,768) INFO - qlib.workflow - [exp.py:258] - Experiment 976138453916236760 starts running ...


[85051:MainThread](2026-10-22 20:13:18,843) INFO - qlib.workflow - [recorder.py:345] - Recorder 0bf6c7fa718f4db6b05c161bf38fe58b starts running under Experiment 976138453916236760 ...


[85051:MainThread](2026-10-22 20:13:19,415) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'recorder_id': '0bf6c7fa718f4db6b05c161bf38fe58b', 'decision': 'diagnostic_only_no_model_change_no_promotion', 'nas_sha256': '95391d2dcd4100bdb75c5378bbfeb2c54ccfe85df91d64d71d45e796ce2ba4c9'}


## 中文最终结论

执行输出见同卡 `conclusion.md` 与 `summary.json`。本卡完成的是 Quant-Arena 式评测桥的**方法接入和现役模型只读研究**：完整账户结果优先，信号统计仅作诊断；没有发现可从该论文直接新增的 SOXL/SOXX 因子，也没有修改现役模型或 SIM。